# Phase-3 Step 1: Load & Explore Mendeley Dataset

**Dataset:** Lithium-ion battery degradation dataset based on a realistic forklift operation profile

**Structure:**
- Cell1/, Cell2/, Cell3/
- Each cell has Round00, Round01, ... (aging rounds)
- Each round has RPT.csv with columns: Cell, Round, Time, Current, Voltage, Energy, Temperature

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os
import warnings
warnings.filterwarnings('ignore')



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("✅ Setup complete")

## 1. Locate Mendeley Data

In [ ]:
# Your Mendeley dataset path
MENDELEY_PATH = os.path.join(
    PROJECT_ROOT, 
    'MENDELEY_Lithium-ion battery degradation dataset based on a realistic forklift operation profile'
)

if os.path.exists(MENDELEY_PATH):
    print(f"✅ Mendeley dataset found!")
    print(f"   Path: {MENDELEY_PATH}")
    
    # List contents
    contents = os.listdir(MENDELEY_PATH)
    print(f"   Contents: {[c for c in contents if not c.startswith('.')]}")
else:
    print(f"❌ Mendeley dataset not found at: {MENDELEY_PATH}")

## 2. Explore Dataset Structure

In [ ]:
from mendeley_loader import explore_mendeley_dataset

info = explore_mendeley_dataset(MENDELEY_PATH)

print("📊 Mendeley Dataset Structure")
print("=" * 50)
print(f"\nCells found: {len(info['cells'])}")

for cell in info['cells']:
    print(f"  {cell['id']}: {cell['n_rounds']} rounds")

print(f"\nTotal rounds: {info['total_rounds']}")
print(f"\nCSV columns: {info['sample_columns']}")

In [ ]:
# Load sample data
sample_file = os.path.join(MENDELEY_PATH, 'Cell1', 'Round00', 'RPT.csv')
sample_df = pd.read_csv(sample_file)

print(f"Sample file: {sample_file}")
print(f"Shape: {sample_df.shape}")
print(f"\nFirst 10 rows:")
display(sample_df.head(10))

print(f"\nData types:")
print(sample_df.dtypes)

print(f"\nStatistics:")
display(sample_df.describe())

## 3. Load All Cells

In [ ]:
from mendeley_loader import MendeleyDatasetLoader

# Load entire dataset
dataset = MendeleyDatasetLoader(MENDELEY_PATH)
cells = dataset.load_all_cells()

In [ ]:
# Get summary
summary = dataset.get_dataset_summary()

print("\n" + "="*50)
print("MENDELEY DATASET SUMMARY")
print("="*50)

print(f"\nTotal cells: {summary['n_cells']}")
print(f"Total cycles: {summary['n_total_cycles']}")

print(f"\nPer-cell details:")
for cell_id, info in summary['cells'].items():
    print(f"\n  {cell_id}:")
    print(f"    Cycles: {info['n_cycles']}")
    print(f"    Rounds: {info['n_rounds']}")
    print(f"    Initial capacity: {info['initial_capacity']:.4f} Ah")
    print(f"    SOH range: {info['soh_range'][0]:.1f}% - {info['soh_range'][1]:.1f}%")

## 4. Plot SOH Degradation

In [ ]:
# Get combined SOH data
soh_df = dataset.get_combined_soh()
print(f"Combined SOH data: {len(soh_df)} samples")
display(soh_df.head())

In [ ]:
# Plot SOH degradation for each cell
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ['blue', 'green', 'red']

for i, cell_id in enumerate(sorted(soh_df['cell_id'].unique())):
    ax = axes[i]
    cell_data = soh_df[soh_df['cell_id'] == cell_id]
    
    ax.plot(cell_data['cycle_number'], cell_data['soh'], 
            color=colors[i], linewidth=1.5, marker='o', markersize=2)
    
    ax.set_xlabel('Cycle Number')
    ax.set_ylabel('SOH (%)')
    ax.set_title(f'{cell_id} SOH Degradation', fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 105)

plt.suptitle('Mendeley Dataset: SOH Degradation Curves', fontsize=14, fontweight='bold')
plt.tight_layout()

os.makedirs('results/phase3', exist_ok=True)
plt.savefig('results/phase3/mendeley_soh_degradation.png', dpi=150)
plt.show()

print("✅ Saved: results/phase3/mendeley_soh_degradation.png")

In [ ]:
# Plot sample discharge cycles
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, cell_id in enumerate(['Cell1', 'Cell2', 'Cell3']):
    if cell_id not in cells:
        continue
    
    loader = cells[cell_id]
    cycles = loader.cycles
    
    if len(cycles) < 2:
        continue
    
    # Voltage vs Time
    ax = axes[0, i]
    # Plot first and last cycle
    first_cycle = cycles[0]
    last_cycle = cycles[-1]
    
    ax.plot(first_cycle.time, first_cycle.voltage, 'b-', label=f'Cycle 0', alpha=0.8)
    ax.plot(last_cycle.time, last_cycle.voltage, 'r-', label=f'Cycle {len(cycles)-1}', alpha=0.8)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Voltage (V)')
    ax.set_title(f'{cell_id}: Voltage Profile', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Capacity per round
    ax = axes[1, i]
    round_df = loader.get_round_summary()
    ax.bar(round_df['round'], round_df['mean_capacity'], color=colors[i], alpha=0.7)
    ax.set_xlabel('Round')
    ax.set_ylabel('Mean Capacity (Ah)')
    ax.set_title(f'{cell_id}: Capacity per Round', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Mendeley Dataset: Discharge Characteristics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/phase3/mendeley_discharge_profiles.png', dpi=150)
plt.show()

print("✅ Saved: results/phase3/mendeley_discharge_profiles.png")

## 5. Extract Features (Same as NASA)

In [ ]:
def extract_cycle_features(cycle, initial_capacity):
    """Extract features from a Mendeley cycle."""
    voltage = cycle.voltage
    current = cycle.current
    temperature = cycle.temperature
    time = cycle.time
    
    # SOH
    soh = (cycle.capacity / initial_capacity) * 100 if initial_capacity > 0 else 100
    
    # Voltage stats
    voltage_mean = np.mean(voltage)
    voltage_std = np.std(voltage)
    voltage_min = np.min(voltage)
    voltage_max = np.max(voltage)
    voltage_range = voltage_max - voltage_min
    
    # Current stats
    current_mean = np.mean(current)
    current_std = np.std(current)
    
    # Temperature stats
    temp_mean = np.mean(temperature)
    temp_max = np.max(temperature)
    temp_delta = np.max(temperature) - np.min(temperature)
    temp_std = np.std(temperature)
    
    # ICA features
    try:
        dv = np.diff(voltage)
        dv[dv == 0] = 1e-6
        dq = np.abs(np.diff(current)) * np.diff(time) / 3600
        ica = dq / np.abs(dv)
        ica = np.clip(ica, 0, 100)
        
        ica_peak_voltage = voltage[np.argmax(ica) + 1] if len(ica) > 0 else 0
        ica_peak_magnitude = np.max(ica) if len(ica) > 0 else 0
        ica_mean = np.mean(ica) if len(ica) > 0 else 0
        ica_std = np.std(ica) if len(ica) > 0 else 0
    except:
        ica_peak_voltage = 0
        ica_peak_magnitude = 0
        ica_mean = 0
        ica_std = 0
    
    # DCIR
    try:
        current_changes = np.abs(np.diff(current))
        voltage_changes = np.abs(np.diff(voltage))
        valid = current_changes > 0.01
        if valid.any():
            dcir = np.median(voltage_changes[valid] / current_changes[valid])
        else:
            dcir = 0
    except:
        dcir = 0
    
    return {
        'battery_id': cycle.cell_id,
        'dataset': 'Mendeley',
        'cycle_number': cycle.cycle_number,
        'round_number': cycle.round_number,
        'soh': min(soh, 100),
        'capacity': cycle.capacity,
        'voltage_mean': voltage_mean,
        'voltage_std': voltage_std,
        'voltage_min': voltage_min,
        'voltage_max': voltage_max,
        'voltage_range': voltage_range,
        'current_mean': current_mean,
        'current_std': current_std,
        'temp_mean': temp_mean,
        'temp_max': temp_max,
        'temp_delta': temp_delta,
        'temp_std': temp_std,
        'ica_peak_voltage': ica_peak_voltage,
        'ica_peak_magnitude': ica_peak_magnitude,
        'ica_mean': ica_mean,
        'ica_std': ica_std,
        'dcir': dcir,
        'discharge_time': time[-1] - time[0] if len(time) > 1 else 0
    }

In [ ]:
# Extract features from all Mendeley cycles
mendeley_features = []

for cell_id, loader in cells.items():
    initial_cap = loader.initial_capacity
    
    for cycle in loader.cycles:
        try:
            features = extract_cycle_features(cycle, initial_cap)
            mendeley_features.append(features)
        except Exception as e:
            print(f"Error processing {cell_id} cycle {cycle.cycle_number}: {e}")

mendeley_df = pd.DataFrame(mendeley_features)
print(f"\n✅ Extracted features for {len(mendeley_df)} Mendeley cycles")
display(mendeley_df.head())

In [ ]:
# Save Mendeley features
os.makedirs('data/phase3', exist_ok=True)
mendeley_df.to_csv('data/phase3/mendeley_features.csv', index=False)
print("✅ Saved: data/phase3/mendeley_features.csv")

## 6. Compare with NASA Dataset

In [ ]:
# Load NASA features
nasa_df = pd.read_csv('data/phase2/harmonized/all_features.csv')
nasa_df['dataset'] = 'NASA'

print(f"NASA samples: {len(nasa_df)}")
print(f"Mendeley samples: {len(mendeley_df)}")
print(f"Total combined: {len(nasa_df) + len(mendeley_df)}")

In [ ]:
# Compare feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

features_to_compare = ['soh', 'capacity', 'voltage_mean', 'temp_mean', 'voltage_range', 'dcir']

for ax, feat in zip(axes.flat, features_to_compare):
    if feat in nasa_df.columns and feat in mendeley_df.columns:
        ax.hist(nasa_df[feat], bins=30, alpha=0.5, label='NASA', density=True)
        ax.hist(mendeley_df[feat], bins=30, alpha=0.5, label='Mendeley', density=True)
        ax.set_xlabel(feat)
        ax.set_ylabel('Density')
        ax.legend()
        ax.set_title(f'{feat} Distribution')
        ax.grid(True, alpha=0.3)

plt.suptitle('NASA vs Mendeley: Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/phase3/nasa_vs_mendeley_distributions.png', dpi=150)
plt.show()

print("✅ Saved: results/phase3/nasa_vs_mendeley_distributions.png")

## 7. Create Combined Dataset

In [ ]:
# Align columns
common_cols = ['battery_id', 'dataset', 'cycle_number', 'soh', 'capacity',
               'voltage_mean', 'voltage_std', 'voltage_min', 'voltage_max', 'voltage_range',
               'current_mean', 'current_std',
               'temp_mean', 'temp_max', 'temp_delta', 'temp_std',
               'ica_peak_voltage', 'ica_peak_magnitude', 'ica_mean', 'ica_std',
               'dcir', 'discharge_time']

# Add missing columns if needed
for col in common_cols:
    if col not in nasa_df.columns:
        nasa_df[col] = 0
    if col not in mendeley_df.columns:
        mendeley_df[col] = 0

# Combine
combined_df = pd.concat([
    nasa_df[common_cols],
    mendeley_df[common_cols]
], ignore_index=True)

print(f"\n📊 Combined Dataset:")
print(f"   NASA: {len(nasa_df)} samples")
print(f"   Mendeley: {len(mendeley_df)} samples")
print(f"   Total: {len(combined_df)} samples")

# Save
combined_df.to_csv('data/phase3/combined_nasa_mendeley.csv', index=False)
print("\n✅ Saved: data/phase3/combined_nasa_mendeley.csv")

In [ ]:
print("\n" + "="*70)
print("STEP 1 SUMMARY: MENDELEY DATA LOADED")
print("="*70)

print(f"""
📊 Mendeley Dataset:
   Cells: {len(cells)}
   Total cycles: {len(mendeley_df)}
   
📊 Combined Dataset:
   NASA cycles: {len(nasa_df)}
   Mendeley cycles: {len(mendeley_df)}
   Total: {len(combined_df)}
   
📁 Files Created:
   - data/phase3/mendeley_features.csv
   - data/phase3/combined_nasa_mendeley.csv
   - results/phase3/mendeley_soh_degradation.png
   - results/phase3/mendeley_discharge_profiles.png
   - results/phase3/nasa_vs_mendeley_distributions.png
""")

print("\n✅ Step 1 Complete - Proceed to Step 2: Cross-Attention Training")